# Lab 4: Hosting Strands Agents in Amazon Bedrock AgentCore Runtime

## Overview

**Amazon Bedrock AgentCore Runtime** is a secure, serverless runtime capability that empowers organizations to deploy and scale AI agents and tools, regardless of framework, protocol, or model choice—enabling rapid prototyping, seamless scaling, and accelerated time to market.

In this tutorial, you'll learn how to take a locally-running Strands agent and deploy it to AgentCore Runtime for production use.

### What You'll Learn

- How to create a simple customer support agent with Strands
- How to test your agent locally
- How to prepare your agent for AgentCore Runtime deployment
- How to deploy and invoke your agent in the cloud
- How to monitor and manage your deployed agent

### Lab Objectives

By the end of this lab, you will have:

✅ Created a customer support agent with multiple tools
✅ Tested the agent locally
✅ Deployed the agent to AgentCore Runtime
✅ Invoked the agent via HTTP and SDK

## Prerequisites

- Python 3.10+
- AWS account with appropriate permissions
- Docker or Finch installed and running
- Amazon Bedrock AgentCore SDK
- Strands Agents framework

In [ ]:
# Import required libraries
import os
import json
import boto3
from strands import Agent
from strands.models import BedrockModel

## Step 1: Creating and Testing Your Agent Locally

First, let's create a simple customer support agent using the tools from our lab helpers. We'll start by importing the pre-built tools and creating a basic agent.

## Step 2: Creating Your Local Agent

Let's create a customer support agent using the tools we've defined. This agent will be able to help customers with shipping information, return policies, product details, and order status.

In [ ]:
%%writefile customer_support_agent.py
from strands import Agent
from strands.models import BedrockModel
from scripts.utils import get_ssm_parameter
from lab_helpers.lab1_strands_agent import (
    get_return_policy,
    get_product_info,
    SYSTEM_PROMPT,
)
from lab_helpers.lab1_guardrails import create_or_get_guardrail_resource
from lab_helpers.lab2_memory import (
    CustomerSupportMemoryHooks,
    create_or_get_memory_resource,
    memory_client,
    ACTOR_ID,
    SESSION_ID,
)

# Lab1 import: Get Bedrock Guardrail
guardrail_id, guardrail_version = create_or_get_guardrail_resource()

# Lab1 import: Create the Bedrock model
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
model = BedrockModel(
    model_id=model_id,
    guardrail_id=guardrail_id,
    # guardrail_version=guardrail_version,
    guardrail_trace="enabled",  # Enable trace info for debugging
)

# Lab2 import : Initialize memory via hooks
memory_id = create_or_get_memory_resource()
memory_hooks = CustomerSupportMemoryHooks(
    memory_id, memory_client, ACTOR_ID, SESSION_ID
)


# Lab1 import: Create the agent with all customer support tools
agent = Agent(
    model=model,
    tools=[get_return_policy, get_product_info],
    system_prompt=SYSTEM_PROMPT,
    hooks=[memory_hooks],
)


def invoke_agent(user_input: str) -> str:
    """Invoke the customer support agent with user input"""
    response = agent(user_input)
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    # Test the agent locally
    test_queries = [
        "What's the return policy for electronics?",
        "Tell me about your laptops",
    ]

    for query in test_queries:
        print(f"\n🤔 User: {query}")
        print(f"🤖 Agent: {invoke_agent(query)}")
        print("-" * 80)

Let's test our agent locally to make sure it works correctly:

In [ ]:
!python customer_support_agent.py

## Step 3: Preparing Your Agent for AgentCore Runtime

Now that we've tested our agent locally, let's prepare it for deployment to AgentCore Runtime. We need to:

1. Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
2. Initialize the App with `app = BedrockAgentCoreApp()`
3. Decorate our invocation function with `@app.entrypoint`
4. Let AgentCore Runtime control the execution with `app.run()`

### Creating the Runtime-Ready Agent

Observe the `#### AGENTCORE RUNTIME - LINE i ####` comments to see where is the relevant deployment code added

In [ ]:
%%writefile agent_runtime.py
from bedrock_agentcore.runtime import (
    BedrockAgentCoreApp,
)  ### AGENTCORE RUNTIME - LINE 1 ###
from strands import Agent
from strands.models import BedrockModel
from scripts.utils import get_ssm_parameter
from lab_helpers.lab1_strands_agent import (
    get_return_policy,
    get_product_info,
    SYSTEM_PROMPT,
    MODEL_ID,
)
from lab_helpers.lab1_guardrails import create_or_get_guardrail_resource
from lab_helpers.lab2_memory import (
    CustomerSupportMemoryHooks,
    create_or_get_memory_resource,
    memory_client,
    ACTOR_ID,
    SESSION_ID,
)

# Lab1 import: Get Bedrock Guardrail
guardrail_id, guardrail_version = create_or_get_guardrail_resource()

# Lab1 import: Create the Bedrock model
model = BedrockModel(
    model_id=MODEL_ID,
    guardrail_id=guardrail_id,
    # guardrail_version=guardrail_version,
    guardrail_trace="enabled",  # Enable trace info for debugging
)

# Lab2 import : Initialize memory via hooks
memory_id = create_or_get_memory_resource()
memory_hooks = CustomerSupportMemoryHooks(
    memory_id, memory_client, ACTOR_ID, SESSION_ID
)

# Lab1 import: Create the agent with all customer support tools
agent = Agent(
    model=model,
    tools=[get_return_policy, get_product_info],
    system_prompt=SYSTEM_PROMPT,
    hooks=[memory_hooks],
)

# Initialize the AgentCore Runtime App
app = BedrockAgentCoreApp()  ### AGENTCORE RUNTIME - LINE 2 ###


@app.entrypoint  ### AGENTCORE RUNTIME - LINE 3 ###
def invoke(payload):
    """AgentCore Runtime entrypoint function"""
    user_input = payload.get("prompt", "")

    # Invoke the agent
    response = agent(user_input)
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    app.run()  ### AGENTCORE RUNTIME - LINE 4 ###

## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

- Creates an HTTP server that listens on port 8080
- Implements the required `/invocations` endpoint for processing requests
- Implements the `/ping` endpoint for health checks
- Handles proper content types and response formats
- Manages error handling according to AWS standards

## Step 4: Deploying to AgentCore Runtime

Now let's deploy our agent to AgentCore Runtime using the AgentCore Starter Toolkit.

### Configure the Runtime Deployment

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Initialize the runtime toolkit
boto_session = boto3.session.Session()
region = boto_session.region_name

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    entrypoint="agent_runtime.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="customer_support_agent"
)
print("Configuration completed:", response)

### Launch the Agent

Now let's launch our agent to AgentCore Runtime:

In [ ]:
# Launch the agent (this will build and deploy the container)
launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

### Check Deployment Status

Let's wait for the deployment to complete:

In [ ]:
import time

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

print(f"Final status: {status}")

## Step 5: Invoking Your Deployed Agent

Now that our agent is deployed and ready, let's test it with some customer support queries.

### Using the AgentCore Starter Toolkit

In [ ]:
# Test different customer support scenarios
test_queries = [
    "What's the return policy for electronics?",
    "Tell me about your laptops",
]

for query in test_queries:
    print(f"\n🤔 Testing: {query}")
    response = agentcore_runtime.invoke({"prompt": query})
    print(f"🤖 Response: {response['response']}")
    print("-" * 80)

### Using AWS SDK (boto3)

You can also use the AWS SDK to invoke your agent:

In [ ]:
# Using boto3 to invoke the agent
agentcore_client = boto3.client('bedrock-agentcore', region_name=region)

try:
    boto3_response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=launch_result.agent_arn,
        qualifier="DEFAULT",
        payload=json.dumps({"prompt": "My Iphone is not connecting with the Bluetooth. What should I do?"})
    )
    
    # Process the response
    if "response" in boto3_response:
        response_content = []
        for event in boto3_response["response"]:
            response_content.append(event.decode('utf-8'))
        print("✅ boto3 invocation successful!")
        print(f"Response: {''.join(response_content)}")
except Exception as e:
    print(f"❌ boto3 invocation failed: {str(e)}")

## Congratulations! 🎉

You have successfully:

✅ Created a customer support agent with multiple tools

✅ Tested the agent locally

✅ Deployed the agent to AgentCore Runtime

✅ Invoked the agent using multiple methods (SDK, HTTP, boto3)

### Key Takeaways

1. **AgentCore Runtime** provides a serverless, scalable platform for hosting AI agents
2. **Minimal code changes** are needed to move from local testing to production deployment
3. **Built-in monitoring and logging** help you track agent performance
